In [1]:
# ===================================================
# CELL 1: SETUP AND IMPORTS
# ===================================================
import os
import sys
import logging
import warnings
import gc
import hashlib
import json
from typing import Tuple, List, Optional, Dict, Any
from pathlib import Path
from dataclasses import dataclass, field
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from tqdm import tqdm
import cv2
from PIL import Image, ImageOps
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow.keras import mixed_precision
import matplotlib.pyplot as plt

# Configure environment
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print("="*70)
print("ENVIRONMENT VERIFICATION")
print("="*70)
print(f"Python:         {sys.version.split()[0]}")
print(f"TensorFlow:     {tf.__version__}")
print(f"NumPy:          {np.__version__}")
print(f"OpenCV:         {cv2.__version__}")
print("-"*70)

# GPU Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU memory growth enabled: {len(gpus)} GPU(s)")
        
        gpu_details = tf.config.experimental.get_device_details(gpus[0])
        print(f"   Device: {gpu_details.get('device_name', 'Unknown')}")
        print(f"   Compute: {gpu_details.get('compute_capability', 'Unknown')}")
        
        # Mixed precision
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        print("✅ Mixed precision enabled (float16)")
    except Exception as e:
        logger.warning(f"GPU setup: {e}")
else:
    print("⚠️ No GPU detected - using CPU")

# Check OpenCV saliency
if hasattr(cv2, 'saliency'):
    print("✅ OpenCV saliency module available")
else:
    print("⚠️ OpenCV saliency module not found")

print("="*70 + "\n")


ENVIRONMENT VERIFICATION
Python:         3.9.23
TensorFlow:     2.10.1
NumPy:          1.23.5
OpenCV:         4.8.1
----------------------------------------------------------------------
✅ GPU memory growth enabled: 1 GPU(s)
   Device: NVIDIA GeForce RTX 3050 Laptop GPU
   Compute: (8, 6)
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 Laptop GPU, compute capability 8.6


[INFO] Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3050 Laptop GPU, compute capability 8.6


✅ Mixed precision enabled (float16)
✅ OpenCV saliency module available



In [2]:
# ===================================================
# CELL 2: CONFIGURATION
# ===================================================
@dataclass
class Config:
    """Configuration - EDIT THE PATH BELOW"""
    # ✏️ CHANGE THIS TO YOUR DATASET PATH
    raw_root: Path = Path(r"D:\monument_dataset\Indian-monuments")
    out_root: Path = Path(r"D:\monument_dataset\ivp_dataset_processed")
    
    # MEMORY OPTIMIZATION: Maximum image dimension
    max_image_dimension: int = 1024  # REDUCED from 2048 for 16GB RAM
    
    @property
    def saliency_output_path(self) -> Path:
        return self.out_root.parent / "saliency_crops"
    
    @property
    def models_dir(self) -> Path:
        return self.out_root.parent / "models"
    
    @property
    def logs_dir(self) -> Path:
        return self.out_root.parent / "logs"
    
    @property
    def results_dir(self) -> Path:
        return self.out_root.parent / "results"
    
    @property
    def visualizations_dir(self) -> Path:
        return self.out_root.parent / "visualizations"
    
    min_crop_dimension: int = 256
    jpeg_quality: int = 90  # Reduced from 95 for smaller files
    
    def validate_paths(self) -> None:
        if not self.raw_root.exists():
            raise FileNotFoundError(f"Dataset not found: {self.raw_root}")
        if not (self.raw_root / "train").exists() or not (self.raw_root / "test").exists():
            raise FileNotFoundError(f"'train' and 'test' folders required in {self.raw_root}")
    
    def create_directories(self) -> None:
        for dir_path in [self.out_root, self.saliency_output_path, self.models_dir, 
                         self.logs_dir, self.results_dir, self.visualizations_dir]:
            dir_path.mkdir(parents=True, exist_ok=True)
        logger.info("✅ All output directories ready")

# Initialize config
config = Config()
config.validate_paths()
config.create_directories()

print(f"📁 Configuration:")
print(f"   Dataset:     {config.raw_root}")
print(f"   Processed:   {config.out_root}")
print(f"   Max img size: {config.max_image_dimension}px (memory optimized)")
print(f"   Models:      {config.models_dir}")
print(f"   Results:     {config.results_dir}\n")


[INFO] ✅ All output directories ready


📁 Configuration:
   Dataset:     D:\monument_dataset\Indian-monuments
   Processed:   D:\monument_dataset\ivp_dataset_processed
   Max img size: 1024px (memory optimized)
   Models:      D:\monument_dataset\models
   Results:     D:\monument_dataset\results



In [ ]:
# ===================================================
# CELL 3: DATA CLEANING (MEMORY OPTIMIZED)
# ===================================================
class DataCleaner:
    """Clean and organize dataset with memory optimization"""
    
    def __init__(self, config: Config):
        self.config = config
        self.stats = defaultdict(lambda: defaultdict(int))
        self.supported_formats = ('.jpg', '.jpeg', '.png', '.jfif', '.webp', '.bmp', '.tiff')
    
    def clean_single_image(self, src_path: Path, dest_path: Path) -> Tuple[bool, str]:
        try:
            if src_path.stat().st_size < 1024:
                return False, "File too small"
            
            with Image.open(src_path) as img:
                img = ImageOps.exif_transpose(img)
                
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                
                # MEMORY OPTIMIZATION: Resize large images immediately
                if img.width > self.config.max_image_dimension or img.height > self.config.max_image_dimension:
                    if img.width > img.height:
                        new_w = self.config.max_image_dimension
                        new_h = int(img.height * (self.config.max_image_dimension / img.width))
                    else:
                        new_h = self.config.max_image_dimension
                        new_w = int(img.width * (self.config.max_image_dimension / img.height))
                    img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
                
                if img.width < 100 or img.height < 100:
                    return False, "Image dimensions too small"
                
                dest_path.parent.mkdir(parents=True, exist_ok=True)
                img.save(dest_path, "JPEG", quality=self.config.jpeg_quality)
                
                return True, "Success"
        except Exception as e:
            return False, str(e)[:50]
    
    def clean_and_organize_dataset(self) -> Dict[str, Any]:
        logger.info("🧹 Starting Data Cleaning (with memory optimization)")
        results = {'processed': 0, 'failed': 0, 'skipped': 0, 'errors': [], 'resized': 0}
        
        for split in ['train', 'test']:
            split_path = self.config.raw_root / split
            if not split_path.exists():
                continue
            
            logger.info(f"\n✨ Processing '{split}' split...")
            
            # NEW: Iterate through style folders, then monument folders
            for style_dir in split_path.iterdir():
                if not style_dir.is_dir():
                    continue
                
                style_name = style_dir.name
                
                for monument_dir in style_dir.iterdir():
                    if not monument_dir.is_dir():
                        continue
                    
                    monument_name = monument_dir.name
                    image_files = [p for ext in self.supported_formats 
                                  for p in monument_dir.glob(f"*{ext}")]
                    
                    if len(image_files) == 0:
                        continue
                    
                    logger.info(f"   Processing {style_name}/{monument_name}: {len(image_files)} images")
                    
                    with ThreadPoolExecutor() as executor:
                        futures = {}
                        for src_path in image_files:
                            file_hash = hashlib.md5(str(src_path).encode()).hexdigest()[:8]
                            dest_name = f"{src_path.stem}_{file_hash}.jpg"
                            # NEW: Maintain style/monument hierarchy
                            dest_path = self.config.out_root / split / style_name / monument_name / dest_name
                            
                            if dest_path.exists():
                                results['skipped'] += 1
                                continue
                            
                            future = executor.submit(self.clean_single_image, src_path, dest_path)
                            futures[future] = (src_path, style_name)
                        
                        for future in tqdm(as_completed(futures), total=len(futures), 
                                          desc=f"  {style_name}/{monument_name}", unit="img", leave=False):
                            src_path, class_name = futures[future]
                            success, message = future.result()
                            
                            if success:
                                results['processed'] += 1
                                self.stats[split][style_name] += 1
                            else:
                                results['failed'] += 1
                                results['errors'].append({'file': str(src_path), 'error': message})
        
        self._print_summary(results)
        return results
    
    def _print_summary(self, results: Dict[str, Any]) -> None:
        print("\n" + "="*50)
        print("🎉 DATA CLEANING COMPLETE")
        print("="*50)
        print(f"✅ Successfully processed: {results['processed']} images")
        print(f"⏭️  Skipped (existing):     {results['skipped']} images")
        print(f"⚠️  Failed:                {results['failed']} images")
        
        if results['errors'] and len(results['errors']) > 0:
            print(f"\n❌ First 5 errors:")
            for error in results['errors'][:5]:
                print(f"   - {Path(error['file']).name}: {error['error']}")
        
        print("\n📊 Class Distribution (Architectural Styles):")
        for split, classes in self.stats.items():
            print(f"\n{split.upper()}:")
            for class_name, count in sorted(classes.items()):
                print(f"   - {class_name}: {count} images")
        print("="*50 + "\n")

print("Running data cleaning...")
cleaner = DataCleaner(config)
cleaning_results = cleaner.clean_and_organize_dataset()
